# 02 Data Cleaning

This notebook shows how the cleaned modeling table is derived from the raw dataset using the validated preprocessing module.


## Reproducibility Note

This notebook is a lightweight narrative companion. The canonical reproducible workflow lives in `src/`, and generated metrics, figures, leakage audit outputs, explainability artifacts, and fairness reports live in `reports/`. The final recommended model is `xgboost_application.pkl`; behavioral and full-diagnostic models are retained only for monitoring or leakage-diagnostic discussion.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

ROOT = Path.cwd()
REPORTS = ROOT / 'reports'
MODELS = ROOT / 'models'
DATA_PATH = next((ROOT / 'data' / 'raw').glob('*'))
pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid')
def load_json(path: Path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)

def show_image_if_exists(path: Path, width: int = 900):
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f'Missing image: {path}')

from src.data_preprocessing import TARGET_COL, prepare_modeling_table
from src.utils import normalize_target

raw_df = pd.read_excel(DATA_PATH)
clean_df = normalize_target(raw_df, target_col=TARGET_COL)
prepared = prepare_modeling_table(raw_df, target_col=TARGET_COL)
print('Raw shape:', raw_df.shape)
print('Prepared shape:', prepared.shape)


## Cleaning choices

- Target values are normalized into binary integers.
- `CustomerID` and `LoanID` are dropped.
- Loan dates are converted into calendar features (`LoanStartYear`, `LoanStartMonth`, `LoanStartQuarter`).
- The previously suspicious hindsight-style `LoanAgeDays` feature is intentionally excluded.


In [ ]:
display(raw_df[[TARGET_COL, 'CustomerID', 'LoanID']].head())
comparison = pd.DataFrame({
    'raw_columns': pd.Series(raw_df.columns),
    'prepared_columns': pd.Series(prepared.columns),
})
display(comparison.head(20))
print('Any missing target values after normalization:', clean_df[TARGET_COL].isna().sum())


The cleaned table is a modeling-ready view. Notebook work stays lightweight by relying on `src.data_preprocessing` rather than duplicating preprocessing logic inline.
